In [13]:
import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

VPIC_BASE = "https://vpic.nhtsa.dot.gov/api/vehicles"


In [14]:
cars_path = RAW_DIR / "used_cars.csv"
df_cars = pd.read_csv(cars_path)

print("Shape:", df_cars.shape)
df_cars.head()


Shape: (99187, 11)


,Unnamed: 0,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,0,T-Roc,2019,25000,Automatic,13904,Diesel,145,49.6,2.0,VW
1,1,T-Roc,2019,26883,Automatic,4562,Diesel,145,49.6,2.0,VW
2,2,T-Roc,2019,20000,Manual,7414,Diesel,145,50.4,2.0,VW
3,3,T-Roc,2019,33492,Automatic,4825,Petrol,145,32.5,2.0,VW
4,4,T-Roc,2019,22900,Semi-Auto,6500,Petrol,150,39.8,1.5,VW


In [15]:
# later work bruhh
def get_all_makes() -> pd.DataFrame:
    """Fetch the full list of vehicle Makes recognized by NHTSA vPIC."""
    url = f"{VPIC_BASE}/GetAllMakes?format=json"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    results = resp.json()["Results"]
    return pd.DataFrame(results)

df_all_makes = get_all_makes()
print(df_all_makes.shape)
df_all_makes.head()


(12340, 2)


,Make_ID,Make_Name
0,12858,#1 ALPINE CUSTOMS
1,4877,"1/OFF KUSTOMS, LLC"
2,11257,"102 IRONWORKS, INC."
3,12255,12832429 CANADA INC.
4,13053,137 INDUSTRIES INC.


In [10]:
df_all_makes.to_csv(RAW_DIR / "vpic_all_makes.csv", index=False)
print("Saved:", RAW_DIR / "vpic_all_makes.csv")


Saved: ..\data\raw\vpic_all_makes.csv


In [11]:
MAKE_MAP = {
    "VW": "Volkswagen", "vauxhall": "Vauxhall", "merc": "Mercedes-Benz",
    "hyundi": "Hyundai", "ford": "Ford", "toyota": "Toyota",
    "skoda": "Skoda", "BMW": "BMW", "Audi": "Audi",
}
df_cars["Make_clean"] = df_cars["Make"].map(MAKE_MAP)



def get_models_for_make(make_name: str) -> pd.DataFrame:
    """Fetch all Models vPIC has on record for a given Make."""
    url = f"{VPIC_BASE}/GetModelsForMake/{make_name}?format=json"
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    results = resp.json()["Results"]
    out = pd.DataFrame(results)
    out["queried_make"] = make_name
    return out

our_makes = sorted(df_cars["Make_clean"].dropna().unique())
print("Fetching models for:", our_makes)

model_frames = []
for make_name in our_makes:
    frame = get_models_for_make(make_name)
    print(f"  {make_name}: {len(frame)} models returned")
    model_frames.append(frame)
    time.sleep(0.5)  # be a polite API citizen

df_models = pd.concat(model_frames, ignore_index=True)
print("Total rows:", df_models.shape)


Fetching models for: ['Audi', 'BMW', 'Ford', 'Hyundai', 'Mercedes-Benz', 'Skoda', 'Toyota', 'Vauxhall', 'Volkswagen']
  Audi: 56 models returned
  BMW: 258 models returned
  Ford: 168 models returned
  Hyundai: 40 models returned
  Mercedes-Benz: 62 models returned
  Skoda: 0 models returned
  Toyota: 58 models returned
  Vauxhall: 0 models returned
  Volkswagen: 40 models returned
Total rows: (682, 5)


In [12]:
df_models.to_csv(RAW_DIR / "vpic_models_by_make.csv", index=False)
print("Saved:", RAW_DIR / "vpic_models_by_make.csv")


Saved: ..\data\raw\vpic_models_by_make.csv
